In [ ]:
import ipywidgets as widgets

In [ ]:
import itertools
from pathlib import Path

from thesis_data_processing import CONTROLLER_MANAGER_DIAGNOSTIC_NAME_MAPPING
from thesis_data_processing import SYSTEM_DIAGNOSTIC_NAME_MAPPING
from thesis_data_processing.step_response import create_dataframes
from thesis_data_processing.step_response import get_trial_data

In [ ]:
rosbag_folder = Path('~/thesis/measurements') / 'experimental' / 'floor2'
rosbag_folder /= 'step-mmt-R20-2025-10-23T1032'
# rosbag_folder /= 'step-mmt-R10-2025-10-23T1102'
# rosbag_folder /= 'step-srv-R20-2025-10-23T1433'
# rosbag_folder /= 'step-srv-R10-2025-10-23T1411'

rosbag_folder = rosbag_folder.expanduser().resolve()

assert rosbag_folder.exists(), rosbag_folder

print('Folder:', rosbag_folder)

In [ ]:
diagnostics_name_mapping = {}
diagnostics_name_mapping.update(CONTROLLER_MANAGER_DIAGNOSTIC_NAME_MAPPING)
diagnostics_name_mapping.update(SYSTEM_DIAGNOSTIC_NAME_MAPPING)

wheel_names: set[str] = {
    f'{fb_pos}_{side}_wheel_joint'
    for fb_pos, side in itertools.product(('front', 'rear'), ('left', 'right'))
}

state_interfaces: set[str] = {
    f'state_interface.{wheel_name}/velocity'
    for wheel_name in wheel_names
}

command_interfaces: set[str] = {
    f'command_interface.{wheel_name}/velocity'
    for wheel_name in wheel_names
}

names_to_keep: set[str] = state_interfaces | command_interfaces


In [ ]:
max_trails, trials = get_trial_data(rosbag_folder)
data_df, diagnostics_data_df = create_dataframes(
    rosbag_folder,
    wheel_names,
    command_interfaces,
    set(trials),
    max_trails,
    names_to_keep=names_to_keep,
    diagnostics_name_mapping=diagnostics_name_mapping,
)

In [ ]:
step_sizes, trial_numbers, _ = zip(*diagnostics_data_df.columns.to_list())
step_sizes = sorted(set(step_sizes))
trial_numbers = sorted(set(trial_numbers))

step_sizes, trial_numbers


layout = widgets.Layout(width='40%')
step_size_selector = widgets.SelectionSlider(
    options=step_sizes,
    value=step_sizes[-1],
    description='step size', 
    layout=layout,
)
trial_number_selector = widgets.SelectionSlider(
    options=trial_numbers,
    value=trial_numbers[0],
    description='Trial #', 
    layout=layout,
)

display(step_size_selector)
display(trial_number_selector)

In [ ]:
step_size = step_size_selector.value
trial_number = trial_number_selector.value
display(f'Selected Step size {step_size}, trial #{trial_number}')

diagnostics_df = diagnostics_data_df[step_size, trial_number].dropna(how='all')

In [ ]:
diagnostics_df[[
    'Mirte-867B16.cpu-monitor.CPU Load Average',
    'Mirte-867B16.ram-monitor.RAM Load Average',
]].dropna(how='all')

In [ ]:
diagnostics_df[[
    'ros2_control.cm-cm-activity.periodicity.max',
    'ros2_control.cm-cm-activity.periodicity.min',
    'ros2_control.cm-cm-activity.periodicity.average',
    'ros2_control.cm-cm-activity.periodicity.standard_deviation',
]].dropna(how='all')


In [ ]:
columns = [column for column in diagnostics_df.columns if column.startswith('ros2_control.cm-cm')]
display(diagnostics_df[columns].dropna(how='all'))

In [ ]:
columns = [column for column in diagnostics_df.columns if column.startswith('ros2_control.cm-cr')]
display(diagnostics_df[columns].dropna(how='all'))

In [ ]:
columns = [column for column in diagnostics_df.columns if column.startswith('ros2_control.cm-hw-activity') and column.count('.') == 2]
display(diagnostics_df[columns].dropna(how='all'))

In [ ]:
columns = [
    column
    for column in diagnostics_df.columns
    if column.startswith('ros2_control.cm-hw-activity')
    and (not column.endswith('state'))
    and ('arm' in column or 'gripper' in column)
]
display(diagnostics_df[columns].dropna(how='all'))

In [ ]:
columns = [
    column
    for column in diagnostics_df.columns
    if column.startswith('ros2_control.cm-hw-activity')
    and (not column.endswith('state'))
    and (not column.startswith('ros2_control.cm-hw-activity.ros2_'))
]
display(diagnostics_df[columns].dropna(how='all'))

In [ ]:
display(diagnostics_df.columns)

In [ ]:
diagnostics_df